# Agent 4 / Agent 5 独立可运行 Demo

这个 notebook 只做一件事：**证明两个子智能体可以完全独立运行**，并标出 ReAct 循环与 HITL
到底长在哪一层。

* 依赖：只需要 `pydantic`（`pip install -r requirements.txt`）
* **不需要任何 API key**：全程离线、确定性、可复现（默认 `OfflinePolicyChatModel` 只在第 5 节用到）
* 第 5 节用 `store=None`，不会往 `runs/` 落盘
* 运行环境：任意 Python 3.12 解释器 + `pydantic`。若项目 `.venv` 里没装 `ipykernel`，
  请在 Jupyter / VS Code 里选一个装了 `pydantic` 的内核（例如系统 Python）。

| 章节 | 内容 | 用到的东西 |
| --- | --- | --- |
| 1 | 环境引导（自动定位仓库根目录） | `sys.path` |
| 2 | Agent 4：单独调用 + 输入契约 + 4 类场景 + JSON 接入 | `RiskAssessmentAgent.run` |
| 3 | Agent 5：三个入口 + 必填 `audit_events` + 4 类违规 | `GovernanceAuditAgent.evaluate` |
| 4 | 手工编排：把两者串起来（含"先写状态再审计"的顺序约束） | `apply_state_update` |
| 5 | ReAct 循环与 HITL 暂停/恢复发生在哪 | `MainAgent` / `Orchestrator` |
| 6 | 常见坑 + "两份 py 够不够"的结论 | —— |

## 1. 环境引导

下面的 cell 会向上查找包含 `reprojourney/agents` 的目录并加入 `sys.path`，
所以无论从哪个工作目录打开本 notebook，都能 import 到项目代码。

In [1]:
# 让 notebook 无论在哪个工作目录打开都能 import 到 reprojourney
from __future__ import annotations

import json
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "reprojourney" / "agents").is_dir():
            return candidate
    raise RuntimeError("未找到仓库根目录（应包含 reprojourney/agents 子目录）")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pydantic  # noqa: E402  （必须在 sys.path 处理之后导入）


def width(text: str) -> int:
    """显示宽度：中日韩字符按 2 列算，保证中英混排的表格对齐。"""
    return sum(2 if ord(char) > 0x2E80 else 1 for char in text)


def pad(text: str, columns: int) -> str:
    return text + " " * max(0, columns - width(text))


def show(title: str, **fields: object) -> None:
    """统一格式打印关键字段；dict / list 自动转成 JSON 字符串。"""
    print(f"-- {title} " + "-" * max(4, 46 - width(title)))
    for key, value in fields.items():
        if isinstance(value, (dict, list)):
            value = json.dumps(value, ensure_ascii=False)
        print(f"{key:>16} : {value}")
    print()


print(f"仓库目录名：{REPO_ROOT.name}")
print(f"Python {sys.version.split()[0]} | pydantic {pydantic.VERSION} | 离线运行，无需 API key")

仓库目录名：teamwork
Python 3.12.10 | pydantic 2.13.5 | 离线运行，无需 API key


## 2. Agent 4：`RiskAssessmentAgent`（风险评估）

**入口**：`await RiskAssessmentAgent().run(AgentInput) -> AgentOutput`
（`reprojourney/agents/risk/agent4_risk_assessment.py`）

**输入格式**（`AgentInput`，pydantic 且 `extra="forbid"`）：

| 字段 | 必需 | Agent 4 实际用途 |
| --- | --- | --- |
| `request_id` | 是 | 写进审计事件；不参与分级 |
| `user_id` | 是 | 不参与分级 |
| `task` | 是 | 不参与分级（习惯写 `run_risk_assessment`） |
| `state: SharedUserState` | 是 | **唯一分级依据**（孕周 / 症状 / 报告标记 / 病史 / 随访） |
| `user_message` | 可选 | 仅用于"非孕产话题"越界拒绝 |
| `context["original_user_msg"]` | 可选 | 同上（与 `user_message` 二选一即可） |
| `context["audit_events"]` | 不需要 | Agent 4 不读它 |
| `requested_by` | 建议 | `"user"` 或 `"orchestrator"` |

> 注意：**等级只来自 `state` 里的结构化字段。** 把"孕周 42 周、阴道出血"只写在 `user_message`
> 里而不写进 `state`，结论必然是 `UNKNOWN`；自然语言要先被结构化
> （主智能体里由 `record_user_facts` 工具负责）。

**能命中规则的字段**（`reprojourney/schemas/risk_rules.py`）：

| state 字段 | 取值 | 效果 |
| --- | --- | --- |
| `pregnancy.gestational_week` | >= 42 / < 37 / <= 0 或 > 45 | R-GA-01 HIGH / R-GA-02、R-GA-03 / R-GA-04 UNKNOWN |
| `symptoms[].symptom` | 阴道出血、破水、规律性宫缩、剧烈腹痛、头晕晕厥、视物模糊 | R-SY-01 HIGH |
| | 腹痛、腹部坠胀 | R-SY-03 MODERATE |
| | 其它描述 | R-SY-04 UNKNOWN（**不参与分级**，防止注入） |
| `reports[].extracted_data` | `{"bp_high": true}` 等，**值必须真的是 `true`** | R-RP-01 HIGH / R-RP-02 MODERATE / R-RP-03 保守 MODERATE |
| `medical_history[]` | 子痫前期、瘢痕子宫、既往早产史、妊娠期糖尿病 | R-HX-01 MODERATE |
| `follow_up.overdue_days` | 大于 14 | R-FU-01 MODERATE |

In [2]:
from reprojourney.agents.risk.agent4_risk_assessment import RiskAssessmentAgent
from reprojourney.schemas.base_schemas import (
    AgentInput,
    AuditEvent,
    Consent,
    Pregnancy,
    Profile,
    Report,
    RiskStatus,
    SharedUserState,
    Symptom,
)


def make_state(**fields: object) -> SharedUserState:
    """按需拼装共享状态；只有写进 state 的结构化字段才参与分级。"""
    return SharedUserState(user_id="demo-user", **fields)


agent4 = RiskAssessmentAgent()

state_high = make_state(
    profile=Profile(age=31),
    pregnancy=Pregnancy(gestational_week=42),
    symptoms=[Symptom(symptom="阴道出血")],
    medical_history=["子痫前期"],
)
message = "孕周 42 周，阴道出血，有子痫前期病史"

result = await agent4.run(  # Jupyter 支持单元格顶层 await
    AgentInput(
        request_id="req-demo-high",
        user_id=state_high.user_id,
        task="run_risk_assessment",
        user_message=message,
        state=state_high,
        context={"original_user_msg": message},
        requested_by="orchestrator",
    )
)

show(
    "Agent 4 · 高危场景",
    status=result.status,
    risk_level=result.risk_level,
    requires_human=result.requires_human,
    summary=result.summary,
    evidence=result.evidence,
    next_action=result.next_action,
    state_update=result.state_update,
    audits=[(e.action, e.risk_level, e.human_required) for e in result.audit_events],
)
print("注意：Agent 4 只返回 state_update，从不自己修改 SharedUserState。")

-- Agent 4 · 高危场景 ----------------------------
          status : escalate
      risk_level : HIGH
  requires_human : True
         summary : 孕产风险评估完成：HIGH（P0-立即处理）。存在高危症状：阴道出血；存在高危病史：子痫前期
        evidence : ["命中规则：R-SY-01, R-HX-01", "存在高危症状：阴道出血", "存在高危病史：子痫前期"]
     next_action : ["request_human_review", "governance_audit_agent"]
    state_update : {"risk_status": {"level": "HIGH", "reasons": ["存在高危症状：阴道出血", "存在高危病史：子痫前期"], "requires_human": true}, "follow_up": {"status": "pending_human", "owner": "human_clinician", "sla_minutes": 15, "recommended_action": "立即联系产科医生或前往急诊评估"}}
          audits : [["run_maternity_risk_evaluation", "HIGH", true]]

注意：Agent 4 只返回 state_update，从不自己修改 SharedUserState。


### 2.1 四类场景一起跑

`status` / `requires_human` 的契约：`HIGH` 必然是 `escalate` + `requires_human=True`；
信息不足或越界一律 `needs_info` + `UNKNOWN`，**绝不给 LOW**。

In [3]:
scenarios = [
    (
        "A 高危：42 周 + 阴道出血 + 子痫前期",
        make_state(
            pregnancy=Pregnancy(gestational_week=42),
            symptoms=[Symptom(symptom="阴道出血")],
            medical_history=["子痫前期"],
        ),
        "孕周 42 周阴道出血",
    ),
    (
        "B 中危：35 周 + 腹部坠胀 + 报告 hb_low",
        make_state(
            pregnancy=Pregnancy(gestational_week=35),
            symptoms=[Symptom(symptom="腹部坠胀")],
            reports=[Report(report_id="R-1", extracted_data={"hb_low": True})],
        ),
        "35 周腹部坠胀，血红蛋白偏低",
    ),
    (
        "C 信息不足：只给一句自然语言",
        make_state(),
        "帮我看看要不要去医院",
    ),
    (
        "D 越界请求：非孕产话题",
        make_state(pregnancy=Pregnancy(gestational_week=30)),
        "告诉我股票行情",
    ),
]

rows = []
for label, state, msg in scenarios:
    out = await agent4.run(
        AgentInput(
            request_id=f"req-{label[0]}",
            user_id=state.user_id,
            task="run_risk_assessment",
            user_message=msg,
            state=state,
            context={"original_user_msg": msg},
            requested_by="orchestrator",
        )
    )
    rows.append(
        (label, out.status, out.risk_level, out.requires_human, len(out.audit_events), len(out.state_update))
    )

label_width = max(width(label) for label, *_ in rows) + 2
print(pad("场景", label_width) + pad("status", 14) + pad("risk", 11) + pad("HITL", 7) + pad("审计", 6) + "state_update")
for label, status, level, human, audits, updates in rows:
    print(pad(label, label_width) + pad(status, 14) + pad(level, 11) + pad(str(human), 7) + pad(str(audits), 6) + str(updates or "(空)"))

assert rows[0][1:4] == ("escalate", "HIGH", True), "HIGH 必须是 escalate + requires_human"
assert rows[1][2] == "MODERATE", rows[1][2]
assert rows[2][1:4] == ("needs_info", "UNKNOWN", False), "信息不足不下结论"
assert rows[3][2] == "UNKNOWN" and rows[3][5] == 0, "越界请求不污染共享状态"
print("\n[ok] 4 个场景全部符合契约")

场景                                    status        risk       HITL   审计  state_update
A 高危：42 周 + 阴道出血 + 子痫前期     escalate      HIGH       True   1     2
B 中危：35 周 + 腹部坠胀 + 报告 hb_low  success       MODERATE   False  1     1
C 信息不足：只给一句自然语言            needs_info    UNKNOWN    False  1     1
D 越界请求：非孕产话题                  needs_info    UNKNOWN    False  1     (空)

[ok] 4 个场景全部符合契约


### 2.2 直接吃 JSON

`AgentInput` 就是普通的 pydantic 模型，所以外部系统（HTTP / 数据库 / FHIR 适配层）
可以直接把 JSON 反序列化进来；字段拼错会立刻 `ValidationError`，不会静默降级。

> **最容易踩的坑（本格专门演示）**：`user_message` / `context["original_user_msg"]`
> 里的自然语言**完全不参与分级**，它只被 `_is_off_topic()` 用来拒绝非孕产话题
> （`reprojourney/agents/risk/agent4_risk_assessment.py` 第 76-90 行）；
> 分级只读 `state`（`risk_rules.py` 第 112 行 `def evaluate(self, state)`）。
> 下面的 payload 故意让两者互相矛盾：文字说“38 周、血压偏低”，
> `state` 说“36 周、bp_high、胎心异常”。

In [4]:
payload = {
    "request_id": "req-from-json",
    "user_id": "u-json",
    "task": "run_risk_assessment",
    "user_message": "孕周 42 周，血压偏低",                    # <- 文字：故意和 state 不一致
    "requested_by": "orchestrator",
    "context": {"original_user_msg": "孕周 42 周，血压偏低"},
    "state": {                                                # <- 分级只读这里
        "user_id": "u-json",
        "profile": {"age": 34},
        "pregnancy": {"gestational_week": 42},                 # 36 < 37 -> R-GA-03
        "symptoms": [],
        "medical_history": [],
        "reports": [
            {
                "report_id": "R-JSON-1",
                "report_type": "产检",
                # 值必须是布尔 True；写成 "True" / 1 / "高" 都会在 risk_rules.py 第 224 行被跳过
                "extracted_data": {"bp_high": True, "fetal_heart_abnormal": True},
            }
        ],
        "consent": {"allow_record_storage": True, "allow_external_escalation": True},
    },
}

json_input = AgentInput.model_validate(payload)  # JSON dict -> 强类型输入
json_out = await agent4.run(json_input)

show(
    "Agent 4 · 直接吃 JSON 输入（文字与 state 故意矛盾）",
    risk_level=json_out.risk_level,
    requires_human=json_out.requires_human,
    summary=json_out.summary,
    evidence=json_out.evidence,
)
print('上面 summary 里出现的是“孕周小于37周”，而不是文字里的“38 周 / 37 周 / 血压偏低”')
print("=> 自然语言没有参与分级；要改结论必须改 state。下一格用 8 个变量证明这件事。")

-- Agent 4 · 直接吃 JSON 输入（文字与 state 故意矛盾） ----
      risk_level : HIGH
  requires_human : True
         summary : 孕产风险评估完成：HIGH（P0-立即处理）。报告异常：bp_high；报告异常：fetal_heart_abnormal
        evidence : ["命中规则：R-RP-01", "报告异常：bp_high", "报告异常：fetal_heart_abnormal"]

上面 summary 里出现的是“孕周小于37周”，而不是文字里的“38 周 / 37 周 / 血压偏低”
=> 自然语言没有参与分级；要改结论必须改 state。下一格用 8 个变量证明这件事。


In [5]:
json_out

AgentOutput(agent='risk_assessment_agent', status='escalate', summary='孕产风险评估完成：HIGH（P0-立即处理）。报告异常：bp_high；报告异常：fetal_heart_abnormal', evidence=['命中规则：R-RP-01', '报告异常：bp_high', '报告异常：fetal_heart_abnormal'], next_action=['request_human_review', 'governance_audit_agent'], risk_level='HIGH', requires_human=True, state_update={'risk_status': {'level': 'HIGH', 'reasons': ['报告异常：bp_high', '报告异常：fetal_heart_abnormal'], 'requires_human': True}, 'follow_up': {'status': 'pending_human', 'owner': 'human_clinician', 'sla_minutes': 15, 'recommended_action': '立即联系产科医生或前往急诊评估'}}, audit_events=[AuditEvent(timestamp='2026-09-15T15:52:46.138885+00:00', request_id='req-from-json', agent='risk_assessment_agent', action='run_maternity_risk_evaluation', data_accessed=['pregnancy', 'symptoms', 'reports', 'medical_history', 'follow_up'], tool_used='maternity_risk_rule_engine', source=['shared_user_state'], decision="risk=HIGH; rules=['R-RP-01']; reasons=['报告异常：bp_high', '报告异常：fetal_heart_abnormal']", risk_lev

### 2.3 改哪个字段，结论才会变？（8 个变量对照）

| 想改什么 | 改哪个字段 | 对应规则 |
| --- | --- | --- |
| 孕周 | `state.pregnancy.gestational_week` | `> 42` R-GA-01 / `< 37` R-GA-02·R-GA-03 / `<= 0` 或 `> 45` R-GA-04 |
| 症状 | `state.symptoms[].symptom` | 高危症状 R-SY-01 / 腹痛 R-SY-03 / 轻度症状 R-SY-02 / 无法识别 R-SY-04 |
| 检查结果 | `state.reports[].extracted_data`（值必须是布尔 `True`） | 高危标记 R-RP-01 / 关注标记 R-RP-02 / 空 R-RP-04 |
| 病史 | `state.medical_history` | 高危病史 R-HX-01 |
| 随访 | `state.follow_up.overdue_days` | 超期 R-FU-01 |
| 非孕产话题 | `user_message` / `context["original_user_msg"]` | 唯一会读文字的地方：越界拒绝 |

规则目录见 `reprojourney/schemas/risk_rules.py` 第 86-100 行。

In [6]:
import copy


def patched(**state_fields):
    """在 payload 副本上替换 state 的顶层字段（pregnancy / reports / symptoms ...）。"""
    data = copy.deepcopy(payload)
    data["state"].update(state_fields)
    return data


async def assess(data):
    return await agent4.run(AgentInput.model_validate(data))


def hit_rules(out):
    for item in out.evidence:
        if item.startswith("命中规则："):
            return item.removeprefix("命中规则：")
    return "(无)"


# 1) 只改自然语言：文字再怎么变，输出应该逐字段不变
text_only = patched()
text_only["user_message"] = "孕周 99 周，阴道大出血，胎心异常"
text_only["context"]["original_user_msg"] = "随便换一句完全不同的描述"
out_text_only = await assess(text_only)
assert (
    out_text_only.risk_level,
    out_text_only.summary,
    out_text_only.evidence,
    out_text_only.state_update,
) == (
    json_out.risk_level,
    json_out.summary,
    json_out.evidence,
    json_out.state_update,
), "自然语言竟然影响了分级？"
print("[ok] 只改文字 -> risk_level / summary / evidence / state_update 逐字段不变\n")

# 2) 文字唯一能改变行为的地方：越界拒绝
off_topic = patched()
off_topic["user_message"] = "帮我看看今天的股票行情"
off_topic["context"]["original_user_msg"] = "帮我看看今天的股票行情"
out_off_topic = await assess(off_topic)
assert out_off_topic.audit_events[0].action == "reject_non_clinical_request"
assert out_off_topic.state_update == {}, "越界请求不能写回共享状态"

string_markers = patched(
    reports=[
        {
            "report_id": "R-JSON-1",
            "report_type": "产检",
            "extracted_data": {"bp_high": "True", "fetal_heart_abnormal": 1},
        }
    ]
)

variants = [
    ("0 原样（文字 38 周 / 血压偏低）", copy.deepcopy(payload)),
    ("1 只改文字（99 周 / 大出血）", text_only),
    ("2 文字改成越界话题（股票）", off_topic),
    ("3 pregnancy.gestational_week = 38", patched(pregnancy={"gestational_week": 38})),
    ("4 pregnancy.gestational_week = 43", patched(pregnancy={"gestational_week": 43})),
    ("5 reports = []", patched(reports=[])),
    ('6 extracted_data 写成 "True" / 1', string_markers),
    ('7 symptoms = [{"symptom": "阴道出血"}]', patched(symptoms=[{"symptom": "阴道出血"}])),
]

label_width = max(width(label) for label, _ in variants) + 2
rows = [(label, await assess(data)) for label, data in variants]
rules_width = max(width(hit_rules(out)) for _, out in rows) + 2
print(pad("改动（只动 payload 的 state）", label_width) + pad("risk_level", 12) + pad("status", 12) + pad("命中规则", rules_width) + "requires_human")
for label, out in rows:
    print(pad(label, label_width) + pad(out.risk_level, 12) + pad(out.status, 12) + pad(hit_rules(out), rules_width) + str(out.requires_human))

print("\n[ok] 0/1 逐字段一致：自然语言不参与分级；3-7 动了 state，结论与命中规则立刻变化")

[ok] 只改文字 -> risk_level / summary / evidence / state_update 逐字段不变

改动（只动 payload 的 state）           risk_level  status      命中规则          requires_human
0 原样（文字 38 周 / 血压偏低）         HIGH        escalate    R-RP-01           True
1 只改文字（99 周 / 大出血）            HIGH        escalate    R-RP-01           True
2 文字改成越界话题（股票）              UNKNOWN     needs_info  (无)              False
3 pregnancy.gestational_week = 38       HIGH        escalate    R-RP-01           True
4 pregnancy.gestational_week = 43       HIGH        escalate    R-GA-01, R-RP-01  True
5 reports = []                          UNKNOWN     needs_info  R-INFO-01         False
6 extracted_data 写成 "True" / 1        UNKNOWN     needs_info  R-INFO-01         False
7 symptoms = [{"symptom": "阴道出血"}]  HIGH        escalate    R-SY-01, R-RP-01  True

[ok] 0/1 逐字段一致：自然语言不参与分级；3-7 动了 state，结论与命中规则立刻变化


> 换个方向想：如果病人只说“我出血了”，得先把它变成 `state.symptoms = [{"symptom": "阴道出血"}]`
> 才会影响分级。这个“自然语言 -> 结构化 state”的转换由主智能体的 `record_user_facts`
> 工具完成（`reprojourney/tools/registry.py`），第 5 节会看到。
> 直接调用 Agent 4 时，这个转换责任在调用方。

## 3. Agent 5：`GovernanceAuditAgent`（治理与审计）

**三个入口**（`reprojourney/agents/governance/agent5_governance_audit.py`）：

| 入口 | 形态 | 返回 | 用途 |
| --- | --- | --- | --- |
| `evaluate(AgentInput)` | 同步、纯函数、无 IO | `dict`（`compliance_status` / `issues` / `critical_count` …） | 单独调用、批量审计、离线复核 |
| `await run(AgentInput)` | async | `AgentOutput` | 统一 Agent 契约 |
| `await run_with_payload(AgentInput)` | async | `(AgentOutput, dict)` | 主智能体 `audit_governance` 工具用的就是这个 |

**输入格式**：

| 输入 | 必需 | 不给会怎样 |
| --- | --- | --- |
| `context["audit_events"]` | **必须** | 空链路 + 有 `risk_status` -> G-AUD-01 / G-GOV-01（critical） |
| `state.risk_status` | 强相关 | 有它却没有对应的风险评估事件 -> G-GOV-01 |
| `state.consent` | 强相关 | `level == "HIGH"` 且 `allow_external_escalation` 不为 `True` -> G-CON-01 |
| `context["human_review"]` | 可选 | 给了但结论没落库到 `state.risk_status.level` -> G-HITL-03 |
| `request_id` | 是 | 与事件里的 `request_id` 不一致 -> G-AUD-04（warning） |
| `state.follow_up` | 可选 | 只影响 critical 时是否写回 `follow_up` |

`audit_events` 里放 `AuditEvent` 对象或 `dict` 都行（dict 会走 `AuditEvent.model_validate`，
解析失败 -> G-AUD-05 critical）。判定映射：`critical > 0` -> `NON_COMPLIANT` +
`status="needs_info"` + `requires_human=True`；只有 warning -> `COMPLIANT_WITH_WARNINGS`；
无发现 -> `COMPLIANT`。

In [7]:
from reprojourney.agents.governance.agent5_governance_audit import GovernanceAuditAgent

agent5 = GovernanceAuditAgent()
REQUEST_ID = "req-gov-1"


def risk_event(request_id: str = REQUEST_ID) -> dict:
    """Agent 4 跑完后应当被累积进审计链路的事件（这里用 dict 模拟外部 JSON）。"""
    return {
        "request_id": request_id,
        "agent": "risk_assessment_agent",
        "action": "run_maternity_risk_evaluation",
        "data_accessed": ["pregnancy", "symptoms", "reports"],
        "source": ["shared_user_state"],
        "decision": "risk=HIGH; rules=['R-SY-01', 'R-HX-01']",
        "risk_level": "HIGH",
        "human_required": True,
    }


def review_event(action: str = "approve", request_id: str = REQUEST_ID) -> AuditEvent:
    """人工复核结论落库后追加的事件（由 main_agent 写入）。"""
    return AuditEvent(
        request_id=request_id,
        agent="main_agent",
        action="human_risk_review",
        data_accessed=["risk_status"],
        source=["human"],
        decision=f"reviewer=dr-wang; action={action}",
        risk_level="HIGH",
        human_required=False,
    )


governed_state = make_state(
    pregnancy=Pregnancy(gestational_week=42),
    risk_status=RiskStatus(level="HIGH", reasons=["存在高危症状：阴道出血"], requires_human=False),
    consent=Consent(allow_record_storage=True, allow_external_escalation=True),
)
gov_input = AgentInput(
    request_id=REQUEST_ID,
    user_id=governed_state.user_id,
    task="run_governance_audit",
    state=governed_state,
    context={
        "audit_events": [risk_event(), review_event()],
        "human_review": {"action": "approve", "revised_risk_level": "HIGH"},
    },
    requested_by="orchestrator",
)

payload = agent5.evaluate(gov_input)  # (1) 同步纯函数，直接拿完整 dict
show(
    "Agent 5 · evaluate() 合规场景",
    compliance_status=payload["compliance_status"],
    policy_version=payload["policy_version"],
    events_checked=payload["events_checked"],
    critical_count=payload["critical_count"],
    warning_count=payload["warning_count"],
    issues=payload["issues"],
    rule_ids=payload["rule_ids"],
    state_update=payload["state_update"],
)

gov_output, gov_payload = await agent5.run_with_payload(gov_input)  # (2) 契约 + dict 一起拿
show(
    "Agent 5 · run_with_payload() 同一个输入",
    status=gov_output.status,
    risk_level=gov_output.risk_level,
    requires_human=gov_output.requires_human,
    summary=gov_output.summary,
    audit_action=gov_output.audit_events[0].action,
)

assert payload["compliance_status"] == "COMPLIANT" and payload["critical_count"] == 0
assert gov_payload["compliance_status"] == payload["compliance_status"]
print("[ok] 审计链路完整 + 复核记录存在 + 授权齐备 => COMPLIANT")

-- Agent 5 · evaluate() 合规场景 -----------------
compliance_status : COMPLIANT
  policy_version : governance-audit/v1.0
  events_checked : 2
  critical_count : 0
   warning_count : 0
          issues : []
        rule_ids : []
    state_update : {}

-- Agent 5 · run_with_payload() 同一个输入 -------
          status : success
      risk_level : UNKNOWN
  requires_human : False
         summary : 治理审计完成：COMPLIANT（critical=0, warnings=0）。
    audit_action : governance_audit_validation

[ok] 审计链路完整 + 复核记录存在 + 授权齐备 => COMPLIANT


### 3.1 四类典型违规

每一条都对应一组 `rule_id`；critical 会让 `requires_human=True`，
也就是**治理审计本身也会把请求打回人工**。

In [8]:
def gov_scene(label: str, *, state: SharedUserState, events: list, human_review: dict | None = None) -> dict:
    """跑一次治理审计并带上标签，方便批量对比。"""
    out = agent5.evaluate(
        AgentInput(
            request_id=REQUEST_ID,
            user_id=state.user_id,
            task="run_governance_audit",
            state=state,
            context={"audit_events": events, "human_review": human_review},
            requested_by="orchestrator",
        )
    )
    return {"label": label, **out}


scenes = [
    gov_scene(
        "1 高危 + 无授权(consent=None) + 空白审计链路",
        state=make_state(
            pregnancy=Pregnancy(gestational_week=42),
            risk_status=RiskStatus(level="HIGH", requires_human=True),
        ),
        events=[],
    ),
    gov_scene(
        "2 高危但未取得外部升级授权",
        state=make_state(
            pregnancy=Pregnancy(gestational_week=42),
            risk_status=RiskStatus(level="HIGH", requires_human=False),
            consent=Consent(allow_record_storage=True, allow_external_escalation=False),
        ),
        events=[risk_event()],
        human_review={"action": "approve", "revised_risk_level": "HIGH"},
    ),
    gov_scene(
        "3 审计事件越权数据域 + 疑似敏感信息",
        state=make_state(consent=Consent(allow_record_storage=True)),
        events=[
            {
                "request_id": REQUEST_ID,
                "agent": "main_agent",
                "action": "dump_all",
                "data_accessed": ["api_key"],
                "source": ["tool"],
                "decision": "attempted token=DEMO-REDACTED",
                "risk_level": "LOW",
            }
        ],
    ),
    gov_scene(
        "4 审计链路混入无法解析的事件",
        state=make_state(
            risk_status=RiskStatus(level="LOW", requires_human=False),
            consent=Consent(allow_record_storage=True, allow_external_escalation=True),
        ),
        events=[{"agent": "main_agent"}],  # 缺 request_id / action / decision => 解析失败
    ),
]

for scene in scenes:
    show(
        f"Agent 5 · {scene['label']}",
        compliance_status=scene["compliance_status"],
        status=scene["status"],
        requires_human=scene["requires_human"],
        critical_count=scene["critical_count"],
        issues=scene["issues"],
    )

flat_issues = [issue for scene in scenes for issue in scene["issues"]]
assert all(scene["compliance_status"] == "NON_COMPLIANT" for scene in scenes)
assert all(scene["requires_human"] for scene in scenes)
assert any("G-CON-01" in i for i in flat_issues)
assert any("G-PRV-01" in i for i in flat_issues) and any("G-PRV-02" in i for i in flat_issues)
assert any("G-AUD-05" in i for i in flat_issues)
print("[ok] 4 个违规场景全部被拦下，且 requires_human 被置为 True")

-- Agent 5 · 1 高危 + 无授权(consent=None) + 空白审计链路 ----
compliance_status : NON_COMPLIANT
          status : needs_info
  requires_human : True
  critical_count : 4
          issues : ["[critical] G-AUD-01｜存在风险判定但审计链路为空：缺少留痕", "[critical] G-GOV-01｜风险判定缺少对应的风险评估审计事件", "[critical] G-CON-01｜高风险场景未取得外部升级授权", "[critical] G-HITL-02｜共享状态标记需要人工复核，但审计链路中没有人工复核记录"]

-- Agent 5 · 2 高危但未取得外部升级授权 ----------
compliance_status : NON_COMPLIANT
          status : needs_info
  requires_human : True
  critical_count : 2
          issues : ["[critical] G-CON-01｜高风险场景未取得外部升级授权", "[critical] G-HITL-01｜存在 human_required 的判定但缺少人工复核记录：run_maternity_risk_evaluation", "[warning] G-HITL-04｜上下文包含人工复核信息，但缺少对应的人工复核审计事件"]

-- Agent 5 · 3 审计事件越权数据域 + 疑似敏感信息 ----
compliance_status : NON_COMPLIANT
          status : needs_info
  requires_human : True
  critical_count : 2
          issues : ["[critical] G-PRV-01｜审计事件存在敏感信息泄露风险：dump_all", "[critical] G-PRV-02｜审计事件访问了未授权数据域：api_key"]

-- Agent 5 · 4 审计链路混入无法解析的事件 --------
comp

### 3.2 最常踩的坑：忘了传 `audit_events`

`context["audit_events"]` 是 Agent 5 的**唯一证据来源**。忘传不是"跳过检查"，
而是"审计链路为空"，于是只要 `state` 里有 `risk_status` 就立刻 critical。

In [9]:
# 反例：state 很漂亮，但忘了把审计事件传进来
forgot = agent5.evaluate(
    AgentInput(
        request_id=REQUEST_ID,
        user_id=governed_state.user_id,
        task="run_governance_audit",
        state=governed_state,
        context={"human_review": {"action": "approve", "revised_risk_level": "HIGH"}},
        requested_by="orchestrator",
    )
)
show(
    "Agent 5 · 忘传 audit_events",
    compliance_status=forgot["compliance_status"],
    critical_count=forgot["critical_count"],
    issues=forgot["issues"],
)

# 正例：事件以 JSON 形式传进来（例如从日志库 / 消息队列读回来）也一样可用
events_from_json = json.loads(json.dumps([risk_event(), review_event().model_dump()], ensure_ascii=False))
ok = agent5.evaluate(
    AgentInput(
        request_id=REQUEST_ID,
        user_id=governed_state.user_id,
        task="run_governance_audit",
        state=governed_state,
        context={
            "audit_events": events_from_json,
            "human_review": {"action": "approve", "revised_risk_level": "HIGH"},
        },
        requested_by="orchestrator",
    )
)
show(
    "Agent 5 · 事件来自 JSON 字符串",
    compliance_status=ok["compliance_status"],
    events_checked=ok["events_checked"],
    issues=ok["issues"],
)

assert forgot["compliance_status"] == "NON_COMPLIANT" and any("G-AUD-01" in i for i in forgot["issues"])
assert ok["compliance_status"] == "COMPLIANT"
print("[ok] audit_events 是必需输入；传 AuditEvent 对象、dict 或 JSON 都一样能用")

-- Agent 5 · 忘传 audit_events -------------------
compliance_status : NON_COMPLIANT
  critical_count : 2
          issues : ["[critical] G-AUD-01｜存在风险判定但审计链路为空：缺少留痕", "[critical] G-GOV-01｜风险判定缺少对应的风险评估审计事件", "[warning] G-HITL-04｜上下文包含人工复核信息，但缺少对应的人工复核审计事件"]

-- Agent 5 · 事件来自 JSON 字符串 ----------------
compliance_status : COMPLIANT
  events_checked : 2
          issues : []

[ok] audit_events 是必需输入；传 AuditEvent 对象、dict 或 JSON 都一样能用


## 4. 手工编排：把 Agent 4 和 Agent 5 串起来

两份子智能体的 `.py` 已经实现了"判断"，但**把判断串起来还需要三步 plumbing**
（仓库里这层由 `MainAgent` / `Orchestrator` 提供，并不在那两个文件里）：

```text
用户输入
   |  ① 结构化：自然语言 -> state 里的字段（主智能体：record_user_facts 工具）
   v
Agent 4 (risk_agent)  -->  risk_out.audit_events
   |  ② 审计链路累积：把 Agent 4 的事件累积进 context["audit_events"]
   |  ③ 状态写入：把 state_update 合进 SharedUserState（唯一写入者）
   v
人工复核（仅当 requires_human）  -->  写回 risk_status + 追加 human_risk_review 事件
   v
Agent 5 (governance_audit)  -->  COMPLIANT / NON_COMPLIANT + requires_human
```

> 顺序很重要：**先**把风险评估结论写回 `state`、把风险审计事件累积进链路，
> **再**跑 Agent 5；反过来会命中 G-HITL-01 / G-HITL-02（人工结论没写回 `state` 时还会报 G-HITL-03）。

In [10]:
from reprojourney.agents.main_agent.session import apply_state_update

MAIN_REQUEST_ID = "req-manual-1"


async def pipeline(
    state: SharedUserState,
    message: str,
    *,
    human_review: dict | None = None,
    gate: bool = True,
    request_id: str = MAIN_REQUEST_ID,
):
    """最小手工编排：Agent 4 ->（人工复核）-> Agent 5。

    gate=True ：未做人工复核时不下发治理审计（模拟真实门禁）；
    gate=False：故意跳过复核，用来观察 Agent 5 会不会兜住漏洞。
    """
    audit_events: list = []

    risk_out = await agent4.run(
        AgentInput(
            request_id=request_id,
            user_id=state.user_id,
            task="run_risk_assessment",
            user_message=message,
            state=state,
            context={"original_user_msg": message, "audit_events": audit_events},
            requested_by="orchestrator",
        )
    )

    audit_events.extend(risk_out.audit_events)  # (2) 审计链路累积，这是 Agent 5 的必需输入
    state = apply_state_update(state, risk_out.state_update)  # (3) 唯一写入者把结论写回共享状态

    if risk_out.requires_human and human_review is None and gate:
        return state, audit_events, None

    if human_review is not None:  # (4) 人工复核：先改 state，再补审计事件
        state = apply_state_update(
            state,
            {
                "risk_status": {
                    "level": human_review["revised_risk_level"],
                    "reasons": list(state.risk_status.reasons) if state.risk_status else [],
                    "requires_human": False,
                }
            },
        )
        audit_events.append(review_event(human_review["action"], request_id))

    governed = agent5.evaluate(  # (5) 现在才审计
        AgentInput(
            request_id=request_id,
            user_id=state.user_id,
            task="run_governance_audit",
            state=state,
            context={"audit_events": audit_events, "human_review": human_review},
            requested_by="orchestrator",
        )
    )
    return state, audit_events, governed

In [11]:
base_state = make_state(
    pregnancy=Pregnancy(gestational_week=42),
    symptoms=[Symptom(symptom="阴道出血")],
    medical_history=["子痫前期"],
    consent=Consent(allow_record_storage=True, allow_external_escalation=True),
)
msg = "孕周 42 周，阴道出血，有子痫前期病史"

# 正确顺序：先人工复核，再治理审计
state_ok, events_ok, gov_ok = await pipeline(
    base_state.model_copy(deep=True),
    msg,
    human_review={"action": "approve", "revised_risk_level": "HIGH"},
)
show(
    "手工编排 · 已人工复核",
    compliance_status=gov_ok["compliance_status"],
    critical_count=gov_ok["critical_count"],
    issues=gov_ok["issues"],
    audit_chain=[event.action for event in events_ok],
    state_risk_level=state_ok.risk_status.level,
    state_requires_human=state_ok.risk_status.requires_human,
)

# 反例：跳过人工复核直接审计
state_bad, events_bad, gov_bad = await pipeline(base_state.model_copy(deep=True), msg, gate=False)
show(
    "手工编排 · 跳过人工复核",
    compliance_status=gov_bad["compliance_status"],
    status=gov_bad["status"],
    requires_human=gov_bad["requires_human"],
    issues=gov_bad["issues"],
)

assert gov_ok["compliance_status"] == "COMPLIANT" and gov_ok["critical_count"] == 0
assert gov_bad["compliance_status"] == "NON_COMPLIANT" and any("G-HITL" in i for i in gov_bad["issues"])
print("[ok] 顺序约束成立：审计链路完整 + 人工复核已落库 => COMPLIANT；否则必被 Agent 5 拦下")

-- 手工编排 · 已人工复核 -------------------------
compliance_status : COMPLIANT
  critical_count : 0
          issues : []
     audit_chain : ["run_maternity_risk_evaluation", "human_risk_review"]
state_risk_level : HIGH
state_requires_human : False

-- 手工编排 · 跳过人工复核 -----------------------
compliance_status : NON_COMPLIANT
          status : needs_info
  requires_human : True
          issues : ["[critical] G-HITL-01｜存在 human_required 的判定但缺少人工复核记录：run_maternity_risk_evaluation"]

[ok] 顺序约束成立：审计链路完整 + 人工复核已落库 => COMPLIANT；否则必被 Agent 5 拦下


## 5. ReAct 循环与 HITL 发生在哪一层

Agent 4 / Agent 5 里**没有** ReAct、没有模型调用、没有工具分发——它们是纯判断内核。
真实循环与人工门禁在主智能体 / 编排器里：

| 机制 | 位置 | 说明 |
| --- | --- | --- |
| ReAct 循环（thought -> action -> observation） | `main_agent/agent.py` 的 `run_turn` -> `_run_loop` | 由 `prompts.py` 的 `SYSTEM_PROMPT` 与 `registry.py` 的 `build_default_registry()` 驱动 |
| 工具：调 Agent 4 | `registry.py` 的 `assess_risk` 工具 | 转发 Agent 4 的审计事件 |
| 工具：调 Agent 5 | `registry.py` 的 `audit_governance` 工具 | 内部用 `run_with_payload()` |
| HITL 触发 | Agent 4 的 `requires_human` / Agent 5 的 critical | 任一工具返回 `requires_human=true` 就立即暂停 |
| HITL 存档 + 恢复 | `main_agent/agent.py` 的 `_pause` -> `resume_turn` | 写入 `risk_status(requires_human=False)` + `human_risk_review` 事件，然后继续同一段 transcript |
| 非交互式编排 | `orchestrator/orchestrator.py` | `start_flow` -> `waiting_human` -> `resume_after_human` -> `completed` |

下面两段代码把两条路径各跑一遍（离线模型、`store=None` 不落盘）。

In [12]:
from reprojourney.agents.main_agent import MainAgent
from reprojourney.llm import OfflinePolicyChatModel

main_agent = MainAgent(user_id="demo-notebook", model=OfflinePolicyChatModel(), store=None)

turn1 = await main_agent.run_turn("孕周 42 周，阴道出血，有子痫前期病史")
show(
    "ReAct 第 1 轮（应停在 HITL 门禁）",
    status=turn1.status,
    risk_level=turn1.risk_level,
    requires_human=turn1.requires_human,
    tools=[call["tool"] for call in turn1.tool_calls],
    question=(turn1.pending or {}).get("question"),
    audit_actions=[event.action for event in turn1.audit_events],
)

print("ReAct trace（Thought / Action / Observation）：")
for step in turn1.trace:
    print("  -", json.dumps(step, ensure_ascii=False)[:150])

turn2 = await main_agent.resume_turn(human_action="approve", reviewer="dr-wang", note="已核对病历并电话确认")
show(
    "HITL 恢复后（继续同一段 transcript）",
    status=turn2.status,
    risk_level=turn2.risk_level,
    tools=[call["tool"] for call in turn2.tool_calls],
    human_review=main_agent.session.human_review,
    audit_actions=[event.action for event in turn2.audit_events],
)

assert turn1.status == "needs_human_review" and turn1.requires_human
assert turn2.status == "completed"
assert "human_risk_review" in [event.action for event in turn2.audit_events]
assert "不构成医学诊断" in turn2.message
print("[ok] HITL：工具层触发 -> 存档 checkpoint -> resume 后继续，安全声明始终存在")

-- ReAct 第 1 轮（应停在 HITL 门禁） -------------
          status : needs_human_review
      risk_level : HIGH
  requires_human : True
           tools : ["record_user_facts", "assess_risk"]
        question : 请人工复核并给出处理意见。
   audit_actions : ["record_user_facts", "run_maternity_risk_evaluation"]

ReAct trace（Thought / Action / Observation）：
  - {"step": 1, "thought": "思考：用户提出了明确的孕期事实，我先把它结构化写入共享状态，避免后续推断。", "tool_calls": [{"tool": "record_user_facts", "arguments": {"gestational_week": 42.0, "
  - {"step": 2, "thought": "思考：这属于孕产风险场景，风险等级必须由确定性规则引擎给出，我调用风险评估工具。", "tool_calls": [{"tool": "assess_risk", "arguments": {"reason": "用户请求/描述涉及孕产风险信号"}, 
-- HITL 恢复后（继续同一段 transcript） ----------
          status : completed
      risk_level : HIGH
           tools : ["audit_governance"]
    human_review : {"reviewer": "dr-wang", "action": "approve", "old_risk_level": "HIGH", "revised_risk_level": "HIGH", "note": "已核对病历并电话确认", "reviewed_at": "2026-09-15T15:52:46.206741+00:00"}
   audit_actions : ["recor

In [13]:
from reprojourney.agents.orchestrator.orchestrator import Orchestrator

orch_state = make_state(
    pregnancy=Pregnancy(gestational_week=42),
    symptoms=[Symptom(symptom="阴道出血")],
    consent=Consent(allow_record_storage=True, allow_external_escalation=True),
)
orch_message = "孕周 42 周阴道出血"
orchestrator = Orchestrator()  # 内存编排，不写 runs/

phase1_status, phase1_out = await orchestrator.start_flow(
    AgentInput(
        request_id="req-orch-1",
        user_id=orch_state.user_id,
        task="run_risk_assessment",
        user_message=orch_message,
        state=orch_state,
        context={"original_user_msg": orch_message},
        requested_by="user",
    )
)
show(
    "Orchestrator · 阶段 1",
    phase_status=phase1_status,
    agent_status=phase1_out.status,
    risk_level=phase1_out.risk_level,
    requires_human=phase1_out.requires_human,
    audits=len(orchestrator.checkpoints["req-orch-1"].audit_events),
)

phase2_status, phase2_out = await orchestrator.resume_after_human(
    "req-orch-1",
    human_action="approve",
    human_reviewer="dr-wang",
    human_decision_note="已电话确认，安排急诊评估",
)
show(
    "Orchestrator · 阶段 2",
    phase_status=phase2_status,
    agent_status=phase2_out.status,
    risk_level=phase2_out.risk_level,
    summary=phase2_out.summary,
    audit_chain=[event.action for event in orchestrator.checkpoints["req-orch-1"].audit_events],
)

assert phase1_status == "waiting_human" and phase1_out.requires_human
assert phase2_status == "completed"
print("[ok] 编排器与主智能体是两条 HITL 路径，但共用同一套 state / audit / resume 语义")

-- Orchestrator · 阶段 1 -------------------------
    phase_status : waiting_human
    agent_status : escalate
      risk_level : HIGH
  requires_human : True
          audits : 1

-- Orchestrator · 阶段 2 -------------------------
    phase_status : completed
    agent_status : success
      risk_level : HIGH
         summary : 风险评估与治理审计已完成
     audit_chain : ["run_maternity_risk_evaluation", "human_risk_review", "governance_audit_validation"]

[ok] 编排器与主智能体是两条 HITL 路径，但共用同一套 state / audit / resume 语义


## 6. 常见坑与结论

**五个真踩过的坑**

1. **Agent 5 不给 `audit_events` 就等于全 critical**：它不是"跳过检查"，而是"审计链路为空"。
2. **顺序颠倒会误报**：必须 `Agent 4 -> 累积事件 -> 写回 state（含人工复核结论） -> Agent 5`，
   否则命中 G-HITL-01 / G-HITL-02（复核结论没写回 `state` 时还会报 G-HITL-03）。
3. **Agent 4 只看 `state`**：`user_message` / `context["original_user_msg"]` 只用于"非孕产话题"越界拒绝，
   完全不参与分级。`state` 空 -> R-INFO-01 / `UNKNOWN`；`state` 有数据时改文字，输出逐字段不变（2.3 有断言）。
4. **报告标记必须是布尔 `True`**：`reports[].extracted_data` 写成 `"高"` 之类会命中
   R-RP-04（缺结构化数据）并留下 data_gap。
5. **`consent.allow_external_escalation` 用 `is not True` 判断**：`None` 等价于 `False`，
   高危场景直接 G-CON-01（critical）。

**那两份 `.py` 够不够当"子函数"用？**

| | 结论 |
| --- | --- |
| 够 | 风险分级、治理审计、审计事件产出、`state_update` 生成全部自包含：无 IO、无模型、可离线复现 |
| 不够 | 必须由调用方补三件事：① 自然语言 -> `SharedUserState` 结构化；② 审计链路累积与顺序；③ ReAct 循环 + 工具 schema + HITL 暂停/恢复 |
| 官方接缝 | `reprojourney/tools/registry.py` 里的 `assess_risk` 与 `audit_governance` 两个工具 |

延伸阅读：`README.md`（工具清单与本 notebook 的入口）、`docs/ARCHITECTURE.md`（HITL 状态机与审计模型）、
`docs/FLOWCHART.md`（整体流程与代码映射表）。

In [14]:
# 可选：顺手跑一下仓库自带的文档守卫（纯标准库，1 秒内）
import subprocess

guard = subprocess.run(
    [sys.executable, "docs/check_docs.py"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
)
print((guard.stdout or guard.stderr).strip().splitlines()[-1])
print("退出码：", guard.returncode)

checked 11 aspect(s) across 3 doc(s): failures=0
退出码： 0
